# Assignment 3: Pandas Analysis and Parquet

In [ ]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
customers = pd.DataFrame(
    [
        (101, "Ahmed", "Cairo"),
        (102, "Sara", "Giza"),
        (103, "Omar", "Cairo"),
        (104, "Mona", "Alexandria"),
        (105, "Nour", "Giza"),
        (106, "Ali", "Cairo"),
    ],
    columns=["CustomerID", "CustomerName", "City"],
)

products = pd.DataFrame(
    [
        (1, "Laptop", "Electronics", 25000),
        (2, "Mouse", "Accessories", 500),
        (3, "Keyboard", "Accessories", 1200),
        (4, "Monitor", "Electronics", 7000),
        (5, "Headphones", "Accessories", 1800),
    ],
    columns=["ProductID", "ProductName", "Category", "UnitPrice"],
)

orders = pd.DataFrame(
    [
        (1001, 101, 1, 1, "2026-01-05", "Completed"),
        (1002, 102, 2, 3, "2026-01-08", "Completed"),
        (1003, 101, 3, 2, "2026-01-15", "Cancelled"),
        (1004, 103, 4, 1, "2026-02-02", "Completed"),
        (1005, 104, 2, 2, "2026-02-10", "Completed"),
        (1006, 102, 5, 1, "2026-02-18", "Completed"),
        (1007, 105, 3, 1, "2026-03-01", "Completed"),
        (1008, 999, 2, 1, "2026-03-05", "Completed"),
        (1009, 103, 1, None, "2026-03-12", "Completed"),
        (1010, 104, 4, 2, "2026-03-20", "Cancelled"),
        (1002, 102, 2, 3, "2026-01-08", "Completed"),
    ],
    columns=["OrderID", "CustomerID", "ProductID", "Quantity", "OrderDate", "Status"],
)

display(customers)
display(products)
display(orders)

In [ ]:
display(orders.head(5))
display(orders.tail(3))
print("customers shape:", customers.shape)
print("products shape:", products.shape)
print("orders shape:", orders.shape)

In [ ]:
print("Columns:", orders.columns.tolist())
print()
print("Data types:")
print(orders.dtypes)
print()
print("Missing values:")
print(orders.isna().sum())

In [ ]:
unit_price = products["UnitPrice"]
print(type(unit_price))
print("Mean:", unit_price.mean())
print("Min:", unit_price.min())
print("Max:", unit_price.max())

In [ ]:
name_price = products[["ProductName", "UnitPrice"]]
display(name_price)
print(type(name_price))
print("Series shape:", unit_price.shape)
print("DataFrame shape:", name_price.shape)

In [ ]:
display(orders.iloc[:4, :3])

In [ ]:
mask = (orders["Status"] == "Completed") & (orders["Quantity"] >= 2)
display(orders.loc[mask, ["OrderID", "CustomerID", "Quantity"]])

In [ ]:
customers_idx = customers.copy().set_index("CustomerID")
display(customers_idx.loc[[102, 105]])

In [ ]:
display(customers_idx.iloc[[1, 3]])

In [ ]:
products_copy = products.copy()
products_copy["UnitPrice"] = products_copy["UnitPrice"].astype(float)
products_copy.loc[products_copy["Category"] == "Accessories", "UnitPrice"] *= 1.10

print("Changed copy:")
display(products_copy)
print("Original products:")
display(products)
print("Original prices unchanged:", products["UnitPrice"].tolist() == [25000, 500, 1200, 7000, 1800])

In [ ]:
clean_orders = orders.copy()
rows_source = len(clean_orders)
duplicate_count = int(clean_orders["OrderID"].duplicated().sum())
print("Rows before removing duplicates:", rows_source)
print("Duplicate OrderID values:", duplicate_count)

clean_orders = clean_orders.drop_duplicates(subset="OrderID", keep="first")
rows_after_dedup = len(clean_orders)
print("Rows after removing duplicates:", rows_after_dedup)

In [ ]:
clean_orders["OrderDate"] = pd.to_datetime(clean_orders["OrderDate"])
clean_orders["Quantity"] = clean_orders["Quantity"].astype("Int64")
print(clean_orders.dtypes)
print("Rows:", len(clean_orders))

In [ ]:
rejected_mask = clean_orders["Quantity"].isna()
rejected_orders = clean_orders[rejected_mask].copy()
print("Rejected rows:", len(rejected_orders))
display(rejected_orders)

In [ ]:
valid_mask = (clean_orders["Quantity"] > 0).fillna(False).astype(bool)
valid_orders = clean_orders[valid_mask].copy()
print("Rows in valid_orders:", len(valid_orders))
display(valid_orders)

In [ ]:
valid_orders["OrderMonth"] = valid_orders["OrderDate"].dt.strftime("%Y-%m")

row_counts = pd.DataFrame(
    {
        "Step": [
            "Source orders",
            "After removing duplicate OrderID",
            "After separating missing Quantity (valid_orders)",
        ],
        "Rows": [rows_source, rows_after_dedup, len(valid_orders)],
    }
)
display(row_counts)
display(valid_orders)

In [ ]:
print("OrderID unique in valid_orders:", valid_orders["OrderID"].is_unique)
print("CustomerID unique in customers:", customers["CustomerID"].is_unique)
print("ProductID unique in products:", products["ProductID"].is_unique)

In [ ]:
left_merge = valid_orders.merge(
    customers,
    on="CustomerID",
    how="left",
    indicator=True,
    validate="many_to_one",
)

unmatched_orders = left_merge[left_merge["_merge"] == "left_only"]
display(unmatched_orders[["OrderID", "CustomerID"]])

In [ ]:
print("valid_orders rows:", len(valid_orders))
print("left_merge rows:", len(left_merge))
print("One row per valid order:", len(left_merge) == len(valid_orders) and left_merge["OrderID"].is_unique)

In [ ]:
inner_merge = valid_orders.merge(
    customers,
    on="CustomerID",
    how="inner",
    validate="many_to_one",
)
print("Left merge rows:", len(left_merge))
print("Inner merge rows:", len(inner_merge))
print("Difference:", len(left_merge) - len(inner_merge))

In [ ]:
filled_merge = left_merge.copy()
filled_merge[["CustomerName", "City"]] = filled_merge[["CustomerName", "City"]].fillna("Unknown")
filled_merge = filled_merge.drop(columns="_merge")
display(filled_merge)

In [ ]:
enriched_orders = filled_merge.merge(
    products,
    on="ProductID",
    how="left",
    validate="many_to_one",
)
print("Rows:", len(enriched_orders))
display(enriched_orders)

In [ ]:
customer_check = customers.merge(
    valid_orders[["CustomerID"]].drop_duplicates(),
    on="CustomerID",
    how="left",
    indicator=True,
)
customers_without_orders = customer_check[customer_check["_merge"] == "left_only"]
display(customers_without_orders[["CustomerID", "CustomerName"]])

In [ ]:
enriched_orders["Revenue"] = enriched_orders["Quantity"] * enriched_orders["UnitPrice"]
display(enriched_orders)

In [ ]:
completed_orders = enriched_orders[enriched_orders["Status"] == "Completed"].copy()
print("Completed orders:", len(completed_orders))
display(completed_orders)

In [ ]:
city_stats = completed_orders.groupby("City").agg(
    OrderCount=("OrderID", "count"),
    TotalUnits=("Quantity", "sum"),
    TotalRevenue=("Revenue", "sum"),
    AvgRevenuePerOrder=("Revenue", "mean"),
)
display(city_stats)

In [ ]:
category_revenue = (
    completed_orders.groupby("Category")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)
display(category_revenue)

In [ ]:
customer_revenue = (
    completed_orders.groupby(["CustomerID", "CustomerName"], as_index=False)["Revenue"]
    .sum()
    .sort_values("Revenue", ascending=False)
    .reset_index(drop=True)
)
display(customer_revenue)
print("Top three customers:")
display(customer_revenue.head(3))

In [ ]:
overall_revenue = completed_orders["Revenue"].sum()
print("Overall completed revenue:", overall_revenue)
print("City total matches:", np.isclose(city_stats["TotalRevenue"].sum(), overall_revenue))
print("Category total matches:", np.isclose(category_revenue.sum(), overall_revenue))
print("Customer total matches:", np.isclose(customer_revenue["Revenue"].sum(), overall_revenue))

In [ ]:
revenue_pivot = pd.pivot_table(
    completed_orders,
    index="City",
    columns="Category",
    values="Revenue",
    aggfunc="sum",
    fill_value=0,
)
display(revenue_pivot)

In [ ]:
revenue_pivot_margins = pd.pivot_table(
    completed_orders,
    index="City",
    columns="Category",
    values="Revenue",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total",
)
display(revenue_pivot_margins)

grand_total = revenue_pivot_margins.loc["Total", "Total"]
print("Pivot grand total:", grand_total)
print("completed_orders revenue:", completed_orders["Revenue"].sum())
print("Match:", np.isclose(grand_total, completed_orders["Revenue"].sum()))

In [ ]:
quantity_pivot = pd.pivot_table(
    completed_orders,
    index="OrderMonth",
    columns="Category",
    values="Quantity",
    aggfunc="sum",
    fill_value=0,
)
display(quantity_pivot)

In [ ]:
try:
    completed_orders.pivot(index="City", columns="Category", values="Revenue")
except ValueError as error:
    print("Direct pivot failed:", error)

In [ ]:
january = completed_orders[completed_orders["OrderMonth"] == "2026-01"].copy()
february = completed_orders[completed_orders["OrderMonth"] == "2026-02"].copy()
march = completed_orders[completed_orders["OrderMonth"] == "2026-03"].copy()

print("January rows:", len(january))
print("February rows:", len(february))
print("March rows:", len(march))

In [ ]:
combined = pd.concat([january, february, march], ignore_index=True)
combined_sorted = combined.sort_values("OrderID").reset_index(drop=True)
original_sorted = completed_orders.sort_values("OrderID").reset_index(drop=True)

pd.testing.assert_frame_equal(combined_sorted, original_sorted)
print("Recombined data matches completed_orders:", combined_sorted.equals(original_sorted))
display(combined_sorted)

In [ ]:
melted = quantity_pivot.reset_index().melt(
    id_vars="OrderMonth",
    var_name="Category",
    value_name="TotalQuantity",
)
display(melted)

In [ ]:
reshaped = (
    melted[melted["TotalQuantity"] != 0]
    .sort_values(["OrderMonth", "Category"])
    .reset_index(drop=True)
)
display(reshaped)

In [ ]:
print("Reshaped total quantity:", reshaped["TotalQuantity"].sum())
print("completed_orders total quantity:", completed_orders["Quantity"].sum())
print("Match:", reshaped["TotalQuantity"].sum() == completed_orders["Quantity"].sum())

In [ ]:
completed_orders.to_parquet(
    "completed_orders.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False,
)
print("Saved:", os.path.exists("completed_orders.parquet"))

In [ ]:
parquet_orders = pd.read_parquet("completed_orders.parquet", engine="pyarrow")

print("Row count equal:", len(parquet_orders) == len(completed_orders))
print("Column names equal:", parquet_orders.columns.tolist() == completed_orders.columns.tolist())
print("Data types equal:", parquet_orders.dtypes.equals(completed_orders.dtypes))
print("Total Revenue equal:", np.isclose(parquet_orders["Revenue"].sum(), completed_orders["Revenue"].sum()))
print()
print(pd.DataFrame({"original": completed_orders.dtypes.astype(str), "parquet": parquet_orders.dtypes.astype(str)}))

In [ ]:
original_sorted = completed_orders.sort_values("OrderID").reset_index(drop=True)
parquet_sorted = parquet_orders.sort_values("OrderID").reset_index(drop=True)

pd.testing.assert_frame_equal(original_sorted, parquet_sorted)
print("assert_frame_equal passed")

In [ ]:
selected = pd.read_parquet(
    "completed_orders.parquet",
    engine="pyarrow",
    columns=["OrderID", "City", "Revenue"],
)
display(selected)

In [ ]:
completed_orders.to_csv("completed_orders.csv", index=False)

parquet_size = os.path.getsize("completed_orders.parquet")
csv_size = os.path.getsize("completed_orders.csv")
print("Parquet size (bytes):", parquet_size)
print("CSV size (bytes):", csv_size)

In [ ]:
def build_city_summary(data):
    summary = (
        data.groupby("City")
        .agg(
            OrderCount=("OrderID", "count"),
            TotalQuantity=("Quantity", "sum"),
            TotalRevenue=("Revenue", "sum"),
        )
        .reset_index()
        .sort_values("TotalRevenue", ascending=False)
        .reset_index(drop=True)
    )
    return summary


city_summary = build_city_summary(completed_orders)
display(city_summary)

In [ ]:
city_summary.to_parquet(
    "city_summary.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False,
)
print("Saved:", os.path.exists("city_summary.parquet"))

In [ ]:
unmatched_valid_orders = left_merge[left_merge["_merge"] == "left_only"].drop(columns="_merge")

rejected_orders.to_csv("rejected_orders.csv", index=False)
unmatched_valid_orders.to_csv("unmatched_customer_orders.csv", index=False)

print("rejected_orders.csv saved:", os.path.exists("rejected_orders.csv"))
print("unmatched_customer_orders.csv saved:", os.path.exists("unmatched_customer_orders.csv"))

In [ ]:
print("Final Summary")
print("-------------")
print("Source order rows:", len(orders))
print("Duplicate rows removed:", duplicate_count)
print("Rejected quantity rows:", len(rejected_orders))
print("Unmatched valid orders:", len(unmatched_valid_orders))
print("Completed orders:", len(completed_orders))
print("Total completed revenue:", completed_orders["Revenue"].sum())

## Task 10: Concept Questions

**1. Explain label-based `loc` selection and position-based `iloc` selection.**

`loc` selects rows and columns by their labels (index values and column names), and slices with `loc` include the end label. It also accepts boolean masks. `iloc` selects by integer position (starting at 0), and slices with `iloc` exclude the end position. For example, after setting `CustomerID` as the index, `loc[[102, 105]]` selects by customer ID, while `iloc[[1, 3]]` selects the second and fourth rows.

**2. Explain the difference between inner and left joins.**

An inner join returns only rows whose keys exist in both DataFrames. A left join returns all rows from the left DataFrame and fills columns from the right DataFrame with missing values when there is no match. Here, the left join kept order 1008 (unknown customer 999), while the inner join dropped it.

**3. How can duplicate keys in a lookup DataFrame multiply rows during a merge?**

If a key appears more than once in the lookup (right) DataFrame, every matching row on the left is paired with each duplicate, so one left row becomes several output rows. For example, if a ProductID appeared twice in `products`, each order for that product would appear twice, inflating row counts and revenue totals. Using `validate="many_to_one"` catches this problem.

**4. When would you use `groupby` instead of a pivot table?**

Use `groupby` when you want a flat, long-format result, need several different named aggregations per group, need custom or multiple grouping levels for further processing, or want to apply functions such as `transform`, `apply`, or `filter`. Use a pivot table when you want a cross-tab layout with one grouping on rows and another on columns, with optional totals.

**5. Why should cancelled orders and missing quantities be handled before calculating business totals?**

Cancelled orders did not generate revenue, so including them would overstate sales. Missing quantities make revenue unknown and can silently produce missing values or wrong totals; guessing them would invent data. Cleaning first keeps counts, revenue, and reports accurate and reproducible.

**6. What are two benefits of Parquet for analytical workloads?**

First, columnar storage with compression gives smaller files and faster scans, because queries can read only the required columns (column pruning). Second, Parquet stores the schema and data types with the data, so types are preserved exactly and reads are consistent and reliable across tools.